# QC and filter seurat object

In [ ]:
library_load <- suppressMessages(
    
    list(
        
        # Seurat 
        library(Seurat), 
        
        # Data 
        library(tidyverse), 
        
        # Plotting 
        library(ggplot2), 
        library(patchwork), 
        
        # Pyhton compatibility
        library(reticulate)
        
    )
    
)

In [ ]:
# Configure reticulate 
use_condaenv(condaenv='p.3.8.12-FD20220623HSCT', conda="/nobackup/peer/fdeckert/miniconda3/bin/conda", required=NULL)
py_config()

In [ ]:
options(warn=-1)

In [ ]:
random_seed <- 42
set.seed(random_seed)

In [ ]:
# Set working directory to project root
setwd("/research/peer/fdeckert/FD20220623HSCT")

In [ ]:
# Source files
source("plotting_global.R")
source("bin/seurat_qc.R")

# Parameter settings

In [ ]:
# Files 
raw_rds_file <- "data/object/raw.rds"
qc_rds_file <- "data/object/qc.rds"
qc_h5ad_file <- "data/object/qc.h5ad"

# Plotting Theme
ggplot2::theme_set(theme_global_set(size_select=1)) # From project global source()

In [ ]:
readRDS("data/object/qc.rds")

# Import Seurat object

In [ ]:
so_qc <- readRDS(raw_rds_file)

# Set meta data 

In [ ]:
so_qc$cell_id <- colnames(so_qc)

In [ ]:
so_qc$time_point <- factor(so_qc$time_point, levels=c("Baseline", "Tx", "D14", "D100", "GVHD"))
so_qc$hto_demux_class <- factor(so_qc$hto_demux_class, levels=c("Blood", "Skin", "Doublet", "Negative"))

# Compute cell cycle score 

In [ ]:
s.features=cc.genes.updated.2019$s.genes

In [ ]:
g2m.features=cc.genes.updated.2019$g2m.genes

In [ ]:
df <- data.frame(genes=c(s.features, g2m.features), phase=c(rep("S", length(s.features)), rep("G2m", length(g2m.features))))

In [ ]:
write.csv(df, "data/reference/cellcycle/seurat.csv")

In [ ]:
# Compute cell cycle score 
so_qc <- CellCycleScoring(so_qc, s.features=cc.genes.updated.2019$s.genes, g2m.features=cc.genes.updated.2019$g2m.genes, set.ident=FALSE)

colnames(so_qc@meta.data) <- gsub("Phase", "cc_phase_class", colnames(so_qc@meta.data))
colnames(so_qc@meta.data) <- gsub("S.Score", "msS_RNA", colnames(so_qc@meta.data))
colnames(so_qc@meta.data) <- gsub("G2M.Score", "msG2M_RNA", colnames(so_qc@meta.data))
    
so_qc$msCC_diff_RNA <- so_qc$msS_RNA - so_qc$msG2M_RNA
so_qc$cc_phase_class <- factor(so_qc$cc_phase_class, levels=names(color$cc_phase_class))

# CellTypist

In [ ]:
celltypist_low <- read.csv("result/celltypist/immune_all_low.csv", row.names=1)
colnames(celltypist_low) <- "celltypist_predicted_labels_low"

In [ ]:
celltypist_high <- read.csv("result/celltypist/immune_all_high.csv", row.names=1)
colnames(celltypist_high) <- "celltypist_predicted_labels_high"

In [ ]:
so_qc <- AddMetaData(so_qc, celltypist_low)
so_qc <- AddMetaData(so_qc, celltypist_high)

# Solo doublets 

In [ ]:
solo <- read.csv("result/solo/solo.csv", row.names=1)
so_qc <- AddMetaData(so_qc, solo)

In [ ]:
options(repr.plot.width=20, repr.plot.height=5)

ggplot(so_qc@meta.data, aes(x=time_point, fill=solo_label)) + 
    geom_bar() + 
    xlab("sample [counts]") + ylab("") + ggtitle("HTO cell demultiplexing") + 
    facet_grid(~patient_id) + 
    scale_fill_manual(values=c("#132B43", "#56B1F7", "#8c8c8c", "#ffc425")) + 
    scale_y_log10() + 
    theme(
        axis.text.x=element_text(angle=45, vjust=1, hjust=1), 
        legend.position="bottom"
    )

# Set Seurat object QC class

In [ ]:
qc_class_set <- function(so) {
    
    # Set QC thresholds
    so$pMT_RNA_min <- -Inf
    so$pMT_RNA_max <- 15
    
    so$pHB_RNA_min <- -Inf
    so$pHB_RNA_max <- 5
    
    so$nCount_RNA_min <- 1000
    so$nCount_RNA_max <- max(so$nCount_RNA)

    so$qc_class <- ifelse(
        so$nCount_RNA <= so$nCount_RNA_max & 
        so$nCount_RNA > so$nCount_RNA_min &
        so$pMT_RNA <= so$pMT_RNA_max &
        so$pMT_RNA > so$pMT_RNA_min & 
        so$pHB_RNA <= so$pHB_RNA_max &
        so$pHB_RNA > so$pHB_RNA_min & 
        so$celltypist_predicted_labels_low != "Early erythroid" & 
        so$celltypist_predicted_labels_low != "Megakaryocyte-erythroid-mast cell progenitor" & 
        so$celltypist_predicted_labels_low != "Mid erythroid" & 
        so$celltypist_predicted_labels_high != "Erythroid" & 
        so$hto_demux_class %in% c("Blood", "Skin") & 
        so$solo_label == "singlet" & 
        so$solo_mad_label == "singlet", 
        "pass", "fail"
      )
    
    # Set percentile threshold for feature count  
    so_tmp <- subset(so, subset=qc_class=="pass")
    
    so$nFeature_RNA_max <- max(so_tmp$nFeature_RNA)
    so$nFeature_RNA_min <- quantile(so_tmp$nFeature_RNA, 0.01)
    
    # Filter data 
    so$qc_class <- ifelse(
        so$qc_class == "pass" &
        so$nFeature_RNA <= so$nFeature_RNA_max & 
        so$nFeature_RNA > so$nFeature_RNA_min, 
        "pass", "fail"
      )
    

    return(so)

}

In [ ]:
so_qc <- Seurat::SplitObject(so_qc, split.by="sample_name")
so_qc <- lapply(so_qc, qc_class_set)
so_qc <- merge(so_qc[[1]], so_qc[2:length(so_qc)])

# Set Factor level 

In [ ]:
so_qc$time_point <- factor(so_qc$time_point, levels=c("Baseline", "Tx", "D14", "D100", "GVHD"))
so_qc$hto_demux_class <- factor(so_qc$hto_demux_class, levels=c("Blood", "Skin", "Doublet", "Negative"))

# UMI and Feature count 

## Density plot

In [ ]:
density_plot_qc_1 <- density_plot_qc(so=so_qc, title="Density plot UMI count", x=nCount_RNA, xlab="log10(UMI count)", min=nCount_RNA_min, max=nCount_RNA_max)
density_plot_qc_2 <- density_plot_qc(so=so_qc, title="Density plot Feature count", x=nFeature_RNA, xlab="log10(Feature count)", min=nFeature_RNA_min, max=nFeature_RNA_max)
density_plot_qc_3 <- density_plot_qc(so=so_qc, title="Density plot Mt %", x=pMT_RNA, xlab="Mt [%]", min=0, max=pMT_RNA_max, xlim=c(0,25), log10=FALSE)

In [ ]:
options(repr.plot.width=20, repr.plot.height=3*10)
density_plot_qc_1 + density_plot_qc_2 + density_plot_qc_3 + plot_layout(ncol=1) & theme(legend.position="none")

## Scattern plot

In [ ]:
scattern_plot_qc_1 <- scattern_plot_qc(so=so_qc, title="Mitochondrial gene percentage", fill=pMT_RNA)
scattern_plot_qc_2 <- scattern_plot_qc(so=so_qc, title="Hemoglobin gene percentage", fill=pHB_RNA)
scattern_plot_qc_3 <- scattern_plot_qc(so=so_qc, title="Ribsonmal gene percentage", fill=pRP_RNA)
scattern_plot_qc_4 <- scattern_plot_qc(so=so_qc, title="QC class", fill=qc_class) + scale_color_manual(values=c("fail"="#132B43", "pass"="#56B1F7"))
scattern_plot_qc_5 <- scattern_plot_qc(so=so_qc, title="HTO demux class", fill=hto_demux_class) + scale_color_manual(values=c("Doublet"="#132B43", "Negative"="#132B43", "Blood"="#56B1F7", "Skin"="#56B1F7"))
scattern_plot_qc_6 <- scattern_plot_qc(so=so_qc, title="SOLO doublet class", fill=solo_label) + scale_color_manual(values=c("doublet"="#132B43", "singlet"="#56B1F7"))

In [ ]:
options(repr.plot.width=20, repr.plot.height=6*10)
scattern_plot_qc_1 + scattern_plot_qc_2 + scattern_plot_qc_3 + scattern_plot_qc_4 + scattern_plot_qc_5 + scattern_plot_qc_6 + plot_layout(ncol=1) & theme(legend.position="bottom")

## Box plots 

In [ ]:
box_plot_qc_1 <- box_plot_qc(so=so_qc, y=nCount_RNA, fill=time_point, ylab="UMI [count]", ymin=0)
box_plot_qc_2 <- box_plot_qc(so=so_qc, y=nFeature_RNA, fill=time_point, ylab="Feature [count]", ymin=0)
box_plot_qc_3 <- box_plot_qc(so=so_qc, y=pMT_RNA, fill=time_point, ylab="Mt [%]", ymin=0, ymax=100)
box_plot_qc_4 <- box_plot_qc(so=so_qc, y=pHB_RNA, fill=time_point, ylab="Hb [%]", ymin=0, ymax=100)
box_plot_qc_5 <- box_plot_qc(so=so_qc, y=pRP_RNA, fill=time_point, ylab="Rb [%]", ymin=0, ymax=100)

In [ ]:
options(repr.plot.width=20, repr.plot.height=5*7.5)
box_plot_qc_1[[1]] + box_plot_qc_1[[2]] + box_plot_qc_2[[1]]  + box_plot_qc_2[[2]] + 
box_plot_qc_3[[1]] + box_plot_qc_3[[2]] + box_plot_qc_4[[1]]  + box_plot_qc_4[[2]] + 
box_plot_qc_5[[1]] + box_plot_qc_5[[2]] + plot_layout(ncol=1)

# Filter cells by QC class

In [ ]:
so_qc <- subset(so_qc, subset=qc_class=="pass")

In [ ]:
so_qc$time_point <- factor(so_qc$time_point, levels=c("Baseline", "Tx", "D14", "D100", "GVHD"))
so_qc$hto_demux_class <- factor(so_qc$hto_demux_class, levels=c("Blood", "Skin", "Doublet", "Negative"))

# Filter genes 

In [ ]:
cnt <- GetAssayData(so_qc, assay="RNA", slot="counts")
cnt <- cnt[rowSums(cnt>3)>1, ]

In [ ]:
so_qc <- CreateSeuratObject(counts=cnt, meta.data=so_qc@meta.data, project="HSCT")

# HTO demultiplexing 

In [ ]:
options(repr.plot.width=20, repr.plot.height=5)

ggplot(so_qc@meta.data, aes(x=time_point, fill=hto_demux_class)) + 
    geom_bar() + 
    xlab("sample [counts]") + ylab("") + ggtitle("HTO cell demultiplexing") + 
    facet_grid(~patient_id) + 
    scale_fill_manual(values=c("#132B43", "#56B1F7", "#8c8c8c", "#ffc425")) + 
    scale_y_log10() + 
    theme(
        axis.text.x=element_text(angle=45, vjust=1, hjust=1), 
        legend.position="bottom"
    )

# UMAP 

In [ ]:
if(TRUE) {
    
    so_qc <- NormalizeData(so_qc)
    so_qc <- ScaleData(so_qc)
    so_qc <- FindVariableFeatures(so_qc, nfeatures=2000)
    so_qc <- RunPCA(so_qc, npcs=50, verbose=FALSE)
    so_qc <- FindNeighbors(so_qc, dims=1:10, k.param=10, verbose=FALSE)
    so_qc <- FindClusters(so_qc, verbose=FALSE, resolution=1, algorithm=1, group.singletons=TRUE)
    so_qc <- RunUMAP(so_qc, dims=1:20, n.neighbors=20, verbose=FALSE)
    
    saveRDS(so_qc, qc_rds_file) 

} else {
    
    so_qc <- readRDS(qc_rds_file)
    
}

In [ ]:
options(repr.plot.width=30, repr.plot.height=25)

dplot_1 <- dplot(so_qc, reduction="umap", group_by="seurat_clusters", label=TRUE) + ggtitle("Louvain cluster")
dplot_2 <- dplot(so_qc, reduction="umap", group_by="patient_id") + scale_color_manual(values=color$patient_id) + ggtitle("Patient ID")
dplot_3 <- dplot(so_qc, reduction="umap", group_by="time_point") + scale_color_manual(values=color$time_point) + ggtitle("Time point")
dplot_4 <- dplot(so_qc, reduction="umap", group_by="hto_demux_class") + ggtitle("HTO demux")
dplot_7 <- dplot(so_qc, reduction="umap", group_by="digestion_protocol") + scale_color_manual(values=color$digestion_protocol) + ggtitle("Digestion protocol")    
dplot_8 <- dplot(so_qc, reduction="umap", group_by="cc_phase_class") + scale_color_manual(values=color$cc_phase_class) + ggtitle("Cell cycle")
dplot_9 <- dplot(so_qc, reduction="umap", group_by="celltypist_predicted_labels_high") + ggtitle("CellTypist majority")
fplot_2 <- fplot(so_qc, reduction="umap", features="pMT_RNA") + scale_colour_viridis_c(option="inferno") + ggtitle("Mt [%]")
fplot_3 <- fplot(so_qc, reduction="umap", features="pHB_RNA") + scale_colour_viridis_c(option="inferno") + ggtitle("Hb [%]")
fplot_4 <- fplot(so_qc, reduction="umap", features="pRP_RNA") + scale_colour_viridis_c(option="inferno") + ggtitle("Rb [%]")
fplot_5 <- fplot(so_qc, reduction="umap", features="nCount_RNA") + scale_colour_viridis_c(option="inferno") + ggtitle("Total UMI")
fplot_6 <- fplot(so_qc, reduction="umap", features="nFeature_RNA") + scale_colour_viridis_c(option="inferno") + ggtitle("Total Features")

dplot_comb <- dplot_1 + dplot_2 + dplot_3 + dplot_4 + dplot_7 + dplot_8 + dplot_9 + fplot_2 + fplot_3 + fplot_4 + fplot_5 + fplot_6 + plot_layout(ncol=4)
dplot_comb

# Save results

In [ ]:
# Store data as h5ad 
adata <- import("anndata", as="adata", convert=FALSE)
pd <- import("pandas", as="pd", convert=FALSE)
np <- import("numpy", as="np", convert=FALSE)
    
# Transform dgCMatrix to sparse sc_sparse matrix
X <- GetAssayData(so_qc, assay="RNA", slot="counts")    
X <- adata$AnnData(X=X)$X$T

adata <- adata$AnnData(X=X, obs=so_qc@meta.data)
adata$var_names <- rownames(GetAssayData(so_qc, assay="RNA", slot="counts"))

adata$raw <- adata
adata$write_h5ad(qc_h5ad_file)

# Session info

In [ ]:
sessionInfo()